# Multi-GPU Distributed Data Parallel (DDP) Training on Kaggle 2x Tesla T4
## 114M Parameter Banach Contraction LLaMA on WikiText-103

This notebook trains and benchmarks 100M+ parameter models with large effective batch size (`65,536 tokens/step`) across 2x Tesla T4 GPUs:
1. **Baseline**: Standard 3-matrix SwiGLU LLaMA-114M
2. **Champion**: Plain 2-Matrix MLP + Spectral Barrier Regularization LLaMA-114M

Both models are trained dynamically until validation loss plateau.

In [ ]:
# 1. Verify 2x Tesla T4 GPU Environment
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | VRAM: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")

In [ ]:
# 2. Install / Upgrade Dependencies
!pip install -q tiktoken datasets transformers

In [ ]:
# 3. Clone Repository (if running standalone in Kaggle) or use local working directory
import os
if not os.path.exists("llama"):
    !git clone https://github.com/phpproger/great.git
    %cd great
!ls -la

In [ ]:
# 4. Launch Multi-GPU DDP Training with torchrun (2x Tesla T4, utilizes ~11.6 GB VRAM each)
# Effective batch size = 2 GPUs * 12 batch * 4 accum = 96 sequences = 49,152 tokens/step
!torchrun --nproc_per_node=2 train_ddp.py \
    --arch compare \
    --model_size 114m \
    --seq_len 512 \
    --batch_size 12 \
    --grad_accum 4 \
    --lr 4e-4 \
    --min_steps 400 \
    --max_steps 4000 \
    --eval_interval 50 \
    --patience 8 \
    --min_delta 0.001 \
    --output_dir checkpoints_ddp

In [ ]:
# 5. Inspect Checkpoints and Outputs
!ls -lh checkpoints_ddp/